<div class="alert alert-success"><h1>Building a More Robust Deep Learning Model in Python</h1></div>

Deep learning models can often be made more stable, efficient, and generalizable by introducing techniques that address common training pitfalls. Methods like **Batch Normalization** mitigate internal covariate shift, **Gradient Clipping** keeps parameters from exploding, **Early Stopping** helps prevent overfitting, and **Learning Rate Scheduling** fine-tunes the optimization process over time. In this tutorial, we will illustrate how to apply each of these techniques in the context of the MNIST digit classification task using Keras.

## Learning Objectives
By the end of this tutorial, you will:
+ Understand how to use Batch Normalization to stabilize and accelerate training.
+ Learn how to use Gradient Clipping to control exploding gradients.
+ Apply Early Stopping to preserve your best-trained model and prevent overfitting.
+ Explore Learning Rate Scheduling to dynamically adjust the learning rate based on validation performance.
+ Combine all these techniques to build a more robust and accurate deep learning model compared to a baseline approach.


## Prerequisites
Before we begin, ensure you have:
+ Basic knowledge of Python programming (variables, functions, classes).
+ Familiarity with the fundamentals of how to build a deep learning model in Python using Keras.
+ A Python (version 3.x) environment with the `tensorflow`, `keras`, and `matplotlib` packages installed.

<div class="alert alert-success"><h2>1. Import and Preprocess the Data</h2></div>

We start by importing the data. For this tutorial, we'll use the **MNIST dataset**, a classic dataset in the machine learning community. It consists of 70,000 grayscale images of handwritten digits ranging from 0 to 9. Each image is 28 x 28 pixels, and the dataset is divided into 60,000 training images and 10,000 testing images. Our goal will be to develop a model that learns to correctly identify a handritten digit given the image.

In [1]:
from tensorflow import keras

keras.utils.set_random_seed(1234)
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Our deep learning model expects the images as a vector of size 784 (i.e. 28 $\times$ 28). So, let's flatten the images.

In [2]:
train_images = train_images.reshape(60000, 28 * 28)
test_images = test_images.reshape(10000, 28 * 28)

The model also expects the image pixel values scaled. Let's do that as well.

In [3]:
train_images = train_images.astype('float32') / 255
test_images = test_images.astype('float32') / 255

Finally, we also need to one-hot encode the image labels.

In [4]:
num_classes = 10
train_labels = keras.utils.to_categorical(train_labels, num_classes)
test_labels = keras.utils.to_categorical(test_labels, num_classes)

<div class="alert alert-info"><b>Note:</b> To learn more about deep learning and how to build a deep learning model using Keras in Python, refer to  the LinkedIn Learning course titled <b>"Deep Learning with Python: Foundations"</b>.</div>

<div class="alert alert-success"><h2>2. Define the Model with Batch Normalization</h2></div>

Our model consists of an input layer with 784 nodes, two hidden layers with 512 and 128 nodes (respectively), and an output layer with 10 nodes. Between each of the hidden layers, we will normalize the outputs of one layer before feeding them into the next. This is known as **Batch Normalization**. Batch Normalization can stabilize the training of a deep learning model and help it converge faster.

To apply BatchNormalization to a model, we simply include a `BatchNormalization()` layer to the model architecture.

In [5]:
from keras.layers import Input, Dense, BatchNormalization

model = keras.Sequential([
    Input(shape = (784,)),
    Dense(512, activation = 'relu'),
    BatchNormalization(),
    Dense(128, activation = 'relu'),
    BatchNormalization(),
    Dense(10, activation = 'softmax')
])

<div class="alert alert-success"><h2>3. Compile the Model with Gradient Clipping</h2></div>

Now that we've defined our model's architecture, let's compile it by specifying the optimizer, loss function and performance metric to optimize. We choose to use the `Adam()` optimizer for our model. By default, optimizers don’t impose any bounds on gradients. However, large gradients can cause a model’s parameters to fluctuate significantly during training and hamper convergence. **Gradient Clipping** mitigates this issue by limiting the magnitude (or norm) of gradients.

To implement Gradient Clipping, we set the `clipnorm` argument within the `Adam()` optimizer to `1.0`. This ensures that the L2 norm of the gradients do not exceed 1.0 during training.

In [6]:
from keras.optimizers import Adam

model.compile(
    optimizer = Adam(clipnorm = 1.0),
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

Note that we can adjust the `clipnorm` value as we see fit based on our dataset or problem. Alternatively, we could have also used the `clipvalue` argument to clip gradiens by value instead of by norm.